# Realocação de verba: mais MQLs com o mesmo orçamento

```

Realocação da verba de mídia entre praças e canais

Pergunta da Loft: como distribuir o orçamento mensal de mídia entre praças e canais
para gerar mais MQLs e cadastros, tirando verba de onde ela rende pouco?

A Loft ainda não enviou os dados de mídia. Este notebook já trabalha no FORMATO que
a Loft descreveu na reunião (dados agregados, sem usuário):

    data | canal | campanha | grupo_anuncios | uf | cidade | investimento | mqls | cadastros

Dois modos:
  - SIMULADO (padrão): o notebook cria dados de mentira nesse formato, com uma
    "resposta certa" escondida (quanto cada praça rende com mais verba), e mede se o
    modelo descobre essa resposta. É assim que se valida o método sem os dados reais.
  - REAL: defina MIDIA_FILE no .env com o arquivo da Loft nesse formato. O notebook
    pula as checagens contra a resposta certa (que só existe na simulação) e gera a
    recomendação de verba.

Hipótese do formato: cada grupo de anúncios é segmentado para uma cidade (como é
comum em campanhas regionais). Se a Loft tiver grupos nacionais, a cidade pode vir do
relatório por região da Meta/Google ou da cidade do lead no CRM.

Machine learning usado:
  - Regressão de Poisson (modelo linear generalizado para contagens): aprende uma
    curva por praça x canal, com o retorno decrescente da verba estimado a partir de
    canal, tipo de praça e tamanho do mercado.
  - Gradient Boosting com restrição monotônica (mais verba nunca reduz MQL), como
    comparação.
  Baselines: custo médio por MQL de cada praça x canal (o que se usa hoje) e
  custo médio geral.

Validação:
  - previsão das últimas 8 semanas, que o modelo não viu
  - no modo simulado: o modelo acerta o custo do PRÓXIMO MQL de cada praça? E quantos
    MQLs a realocação gera de verdade, comparada com a regra simples e com o máximo
    possível?
  - teste de estresse: saturação sem relação com nenhuma variável e verba quase parada

Respostas aos itens combinados com a Loft:
  - custo do próximo MQL por tipo de praça
  - quanto investimento está em praças de baixo retorno
  - economia: mesmos MQLs com menos verba; ou mais MQLs com a mesma verba

Entradas:
  reports/tables/potencial_cidades.csv (03)  - tamanho do mercado de aluguel por cidade
  DATA_DIR/base_municipios_POC.csv          - tipo de praça (clusterização)
  DATA_DIR/<MIDIA_FILE> (opcional)          - mídia real da Loft

Saídas:
  reports/tables/realocacao_modelo_planilha_loft.csv       (planilha-modelo para a Loft preencher)
  reports/tables/realocacao_midia_formato_loft_SIMULADO.csv (dados simulados, se modo simulado)
  reports/tables/realocacao_comparacao_modelos.csv
  reports/tables/realocacao_recomendacao_praca_canal.csv
  reports/tables/realocacao_por_tipo_de_praca.csv
  reports/tables/realocacao_sensibilidade_limites.csv
  reports/figures/realocacao_*.png
```

## Imports

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# O ajuste da Poisson chega ao limite de iterações com centenas de colunas, mas o
# resultado já está estável (o teste contra a resposta certa confirma); o aviso é silenciado
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## Parâmetros

In [ ]:
SEMENTE = 42
ARQUIVO_MUNICIPIOS = "base_municipios_POC.csv"

# Simulação
CIDADES_POR_TIPO = 16                 # cidades simuladas por tipo de praça (as de maior mercado)
ORCAMENTO_MENSAL_SIMULADO = 1_500_000
DIAS_SIMULADOS = 365
INICIO_SIMULACAO = "2025-10-01"
CANAIS = {
    # canal: (MQL máximo relativo, ponto de saturação relativo, fatia da verba hoje)
    "Meta": (1.0, 1.0, 0.45),
    "Google Search": (1.4, 0.8, 0.35),
    "Google Display": (0.5, 1.5, 0.20),
}
CAMPANHAS = {"Fiança - Imobiliárias": 0.6, "Fiança - Inquilinos": 0.4}   # fatia da verba de cada canal
FATOR_PRACA = {                       # quanto cada tipo de praça rende (só na simulação)
    "Praça madura de locação": 1.3,
    "Praça intermediária conectada": 1.0,
    "Praça popular de grande porte": 0.8,
    "Praça em formação": 0.9,
    "Praça sem mercado formal": 0.4,
}
TAXA_CADASTRO = {                     # fração dos MQLs que viram cadastro (só na simulação)
    "Praça madura de locação": 0.35,
    "Praça intermediária conectada": 0.28,
    "Praça popular de grande porte": 0.22,
    "Praça em formação": 0.25,
    "Praça sem mercado formal": 0.12,
}

# Modelo e otimização
SEMANAS_TESTE = 8
OBJETIVO = "mqls"                     # "mqls" ou "cadastros"
LIMITE_VARIACAO = 0.30                # cada praça x canal pode subir ou cair no máximo 30%
PASSOS_OTIMIZADOR = 30
LIMIAR_ACAO = 0.10                    # variação de 10% ou mais vira AUMENTAR / REDUZIR
SEMANAS_POR_MES = 4.33

COLUNAS_LOFT = ["data", "canal", "campanha", "grupo_anuncios", "uf", "cidade",
                "investimento", "mqls", "cadastros"]

## Caminhos e modo

In [ ]:
# No notebook não existe __file__: o projeto é a pasta acima de notebooks/
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")

caminho_dados = os.getenv("DATA_DIR")
if caminho_dados is None:
    raise ValueError("A variável DATA_DIR não foi encontrada. Confira o arquivo .env.")
DATA_DIR = Path(caminho_dados)

TABLES_DIR = PROJECT_DIR / "reports" / "tables"
FIGURES_DIR = PROJECT_DIR / "reports" / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ARQUIVO_POTENCIAL = TABLES_DIR / "potencial_cidades.csv"
ARQUIVO_MUN = DATA_DIR / os.getenv("MUNICIPIOS_FILE", ARQUIVO_MUNICIPIOS)
ARQUIVO_MIDIA = DATA_DIR / os.getenv("MIDIA_FILE", "") if os.getenv("MIDIA_FILE") else None

SIMULADO = ARQUIVO_MIDIA is None or not ARQUIVO_MIDIA.exists()
SUFIXO = "_SIMULADO" if SIMULADO else ""

for arquivo, dica in [(ARQUIVO_POTENCIAL, "rode antes o notebook 03_potencial_mercado"),
                      (ARQUIVO_MUN, "coloque base_municipios_POC.csv na pasta DATA_DIR")]:
    if not arquivo.exists():
        raise FileNotFoundError(f"{arquivo.name} não encontrado: {dica}.")

print("MODO:", "SIMULADO (dados de mentira com resposta certa escondida)" if SIMULADO
      else f"REAL ({ARQUIVO_MIDIA.name})")

## Planilha-modelo para a loft

In [ ]:
# Arquivo pequeno, com as colunas exatas e linhas inventadas, para a Loft ver o formato
modelo_planilha = pd.DataFrame([
    ["2026-10-05", "Meta", "Fiança - Imobiliárias", "Campinas - Imobiliárias", "SP", "CAMPINAS", 1200.00, 15, 4],
    ["2026-10-05", "Google Search", "Fiança - Imobiliárias", "Campinas - Busca fiança", "SP", "CAMPINAS", 850.50, 12, 3],
    ["2026-10-05", "Meta", "Fiança - Inquilinos", "Curitiba - Inquilinos", "PR", "CURITIBA", 640.00, 6, 1],
], columns=COLUNAS_LOFT)
modelo_planilha.to_csv(TABLES_DIR / "realocacao_modelo_planilha_loft.csv", index=False)
print("Planilha-modelo salva:", TABLES_DIR / "realocacao_modelo_planilha_loft.csv")

## Cidades e suas características (dados públicos + base de anúncios da loft)

In [ ]:
def normalizar_nome(s):
    return (
        s.astype(str).str.normalize("NFKD").str.encode("ascii", errors="ignore")
        .str.decode("ascii").str.upper().str.replace(r"[^A-Z ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True).str.strip()
    )


potencial = pd.read_csv(ARQUIVO_POTENCIAL)
municipios = pd.read_csv(ARQUIVO_MUN, dtype={"id_municipio": str})
potencial["chave"] = normalizar_nome(potencial["cidade"])
municipios["chave"] = normalizar_nome(municipios["municipio"])

perfil_cidades = potencial.merge(
    municipios[["sigla_uf", "chave", "populacao", "tipo_de_praca"]].rename(columns={"sigla_uf": "uf"}),
    on=["uf", "chave"], how="inner",
)[["uf", "cidade", "chave", "populacao", "tipo_de_praca", "anuncios_mes"]]

print("Cidades com mercado (03) e tipo de praça (clusterização):", len(perfil_cidades))

## Dados de mídia: simulados (com resposta certa escondida) ou reais

In [ ]:
def simular(semente, variacao_mensal=0.25, variacao_diaria=0.20, saturacao_ligada_ao_perfil=True):
    """
    Cria dados diários no formato da Loft. Para cada praça x canal escolhe uma curva
    verdadeira: MQL por dia = A x verba / (verba + K)
      A = MQL máximo por dia (cresce com o mercado, o canal e o tipo de praça)
      K = verba diária em que a praça entrega metade do máximo (ponto de saturação)
    A verba de hoje segue a população (e não o potencial), então está mal distribuída.
    """
    rng = np.random.default_rng(semente)
    cidades = (perfil_cidades.sort_values("anuncios_mes", ascending=False)
               .groupby("tipo_de_praca", group_keys=False).head(CIDADES_POR_TIPO)
               .reset_index(drop=True))
    tamanho = cidades["anuncios_mes"] / cidades["anuncios_mes"].median()

    verdade = []
    for i, cid in cidades.iterrows():
        for canal, (fator_a, fator_k, fatia) in CANAIS.items():
            A = 12 * tamanho[i] ** 0.6 * fator_a * FATOR_PRACA[cid["tipo_de_praca"]] * rng.lognormal(0, 0.35)
            if saturacao_ligada_ao_perfil:
                K = 900 * tamanho[i] ** 0.5 * fator_k * rng.lognormal(0, 0.35)
            else:
                K = 900 * tamanho[i] ** 0.5 * rng.lognormal(0, 0.8)
            verdade.append({"uf": cid["uf"], "cidade": cid["cidade"], "canal": canal,
                            "tipo_de_praca": cid["tipo_de_praca"], "A": A, "K": K, "fatia": fatia})
    verdade = pd.DataFrame(verdade)

    peso_pop = (cidades["populacao"] ** 0.9) / (cidades["populacao"] ** 0.9).sum()
    orcamento_dia = ORCAMENTO_MENSAL_SIMULADO / 30
    peso_cidade = verdade["cidade"].map(dict(zip(cidades["cidade"], peso_pop)))
    verdade["verba_base"] = orcamento_dia * peso_cidade * verdade["fatia"] * rng.lognormal(0, 0.3, len(verdade))
    verdade["verba_base"] *= orcamento_dia / verdade["verba_base"].sum()

    dias = pd.date_range(INICIO_SIMULACAO, periods=DIAS_SIMULADOS)
    dia_util = np.where(dias.weekday < 5, 1.1, 0.75)
    indice_mes = (dias.year - dias.year[0]) * 12 + dias.month - dias.month[0]

    linhas = []
    for _, v in verdade.iterrows():
        ajuste_mes = rng.lognormal(0, variacao_mensal, indice_mes.max() + 1)[indice_mes]
        verba = v["verba_base"] * ajuste_mes * rng.lognormal(0, variacao_diaria, len(dias))
        for campanha, fatia_camp in CAMPANHAS.items():
            verba_c = verba * fatia_camp
            mqls = rng.poisson(v["A"] * verba / (verba + v["K"]) * dia_util * fatia_camp)
            cadastros = rng.binomial(mqls, TAXA_CADASTRO[v["tipo_de_praca"]])
            publico = campanha.split(" - ")[1]
            linhas.append(pd.DataFrame({
                "data": dias, "canal": v["canal"], "campanha": campanha,
                "grupo_anuncios": f"{v['cidade'].title()} - {publico}",
                "uf": v["uf"], "cidade": v["cidade"],
                "investimento": verba_c.round(2), "mqls": mqls, "cadastros": cadastros,
            }))
    return pd.concat(linhas, ignore_index=True), verdade


if SIMULADO:
    midia, verdade = simular(SEMENTE)
    midia.to_csv(TABLES_DIR / "realocacao_midia_formato_loft_SIMULADO.csv", index=False)
else:
    midia = pd.read_csv(ARQUIVO_MIDIA, parse_dates=["data"])
    faltando = set(COLUNAS_LOFT) - set(midia.columns)
    if faltando:
        raise ValueError(f"O arquivo da Loft não tem as colunas: {sorted(faltando)}")
    midia["cidade"] = normalizar_nome(midia["cidade"])
    midia["uf"] = midia["uf"].str.upper().str.strip()
    verdade = None

midia["data"] = pd.to_datetime(midia["data"])
print("\nLinhas no formato da Loft:", len(midia))
print("Período:", midia["data"].min().date(), "a", midia["data"].max().date())
print("Canais:", sorted(midia["canal"].unique()))
print("Campanhas:", midia["campanha"].nunique(), "| Grupos de anúncios:", midia["grupo_anuncios"].nunique(),
      "| Cidades:", midia["cidade"].nunique())
meses_no_arquivo = midia["data"].dt.to_period("M").nunique()
print(f"Investimento médio por mês: R$ {midia['investimento'].sum() / meses_no_arquivo:,.0f}")
print(f"MQLs por mês: {midia['mqls'].sum() / meses_no_arquivo:,.0f} | "
      f"Custo médio por MQL: R$ {midia['investimento'].sum() / midia['mqls'].sum():,.2f}")
print("\nPrimeiras linhas:")
print(midia.head().to_string(index=False))

## Base do modelo: praça x canal x semana

In [ ]:
# Campanha e grupo de anúncios são somados dentro de cada praça x canal: a decisão de
# verba é tomada nesse nível, e a semana reduz o ruído do dia a dia
midia["semana"] = midia["data"].dt.to_period("W").dt.start_time
semanal = (
    midia.groupby(["uf", "cidade", "canal", "semana"], as_index=False)[["investimento", "mqls", "cadastros"]]
    .sum()
)
semanal = semanal[semanal["investimento"] > 0].copy()

semanal["chave"] = normalizar_nome(semanal["cidade"])
semanal = semanal.merge(perfil_cidades[["uf", "chave", "tipo_de_praca", "anuncios_mes"]],
                        on=["uf", "chave"], how="left")
semanal["tipo_de_praca"] = semanal["tipo_de_praca"].fillna("Sem classificação")
semanal["anuncios_mes"] = semanal["anuncios_mes"].fillna(perfil_cidades["anuncios_mes"].median())

semanal["praca_canal"] = semanal["cidade"] + " | " + semanal["canal"]
semanal["log_investimento"] = np.log(semanal["investimento"])

celulas = sorted(semanal["praca_canal"].unique())
canais = sorted(semanal["canal"].unique())
tipos = sorted(semanal["tipo_de_praca"].unique())

semanas = sorted(semanal["semana"].unique())
teste = (semanal["semana"] >= semanas[-SEMANAS_TESTE]).to_numpy()
treino = ~teste

print(f"\nPraças x canais: {len(celulas)} | Semanas: {len(semanas)} "
      f"(treino {len(semanas) - SEMANAS_TESTE}, teste {SEMANAS_TESTE})")

## Modelos

In [ ]:
def matriz_poisson(df):
    """
    log(MQL esperado) = nível da praça x canal + elasticidade x log(verba)
    A elasticidade (quanto o resultado reage a +1% de verba) muda com canal,
    tipo de praça e tamanho do mercado.
    """
    colunas = {f"nivel={c}": (df["praca_canal"] == c).astype(float) for c in celulas}
    colunas["log_verba"] = df["log_investimento"]
    for v in canais:
        colunas[f"log_verba x canal={v}"] = df["log_investimento"] * (df["canal"] == v)
    for v in tipos:
        colunas[f"log_verba x praca={v}"] = df["log_investimento"] * (df["tipo_de_praca"] == v)
    colunas["log_verba x log_mercado"] = df["log_investimento"] * np.log(df["anuncios_mes"])
    return pd.DataFrame(colunas, index=df.index)


def matriz_boosting(df):
    return pd.DataFrame({
        "praca_canal": df["praca_canal"].map({c: i for i, c in enumerate(celulas)}),
        "log_investimento": df["log_investimento"],
        "canal": df["canal"].map({c: i for i, c in enumerate(canais)}),
        "tipo_de_praca": df["tipo_de_praca"].map({c: i for i, c in enumerate(tipos)}),
        "log_mercado": np.log(df["anuncios_mes"]),
    }, index=df.index)


def treinar(alvo):
    poisson = PoissonRegressor(alpha=1e-5, max_iter=5000)
    poisson.fit(matriz_poisson(semanal[treino]), semanal.loc[treino, alvo])
    # O boosting só trata 'praca_canal' como categoria se houver até 255 delas
    cat_praca = len(celulas) <= 255
    boosting = HistGradientBoostingRegressor(
        loss="poisson", categorical_features=[cat_praca, False, True, True, False],
        monotonic_cst=[0, 1, 0, 0, 0],      # mais verba nunca reduz o resultado
        max_iter=800, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,
        l2_regularization=1.0, random_state=SEMENTE,
    )
    boosting.fit(matriz_boosting(semanal[treino]), semanal.loc[treino, alvo])
    return poisson, boosting


modelos = {alvo: treinar(alvo) for alvo in ["mqls", "cadastros"]}

## Validação: previsão das últimas semanas

In [ ]:
def custo_medio_celula(alvo):
    t = semanal[treino].groupby("praca_canal")[["investimento", alvo]].sum()
    return t["investimento"] / t[alvo].clip(lower=1)


resultados = []
for alvo in ["mqls", "cadastros"]:
    poisson, boosting = modelos[alvo]
    real = semanal.loc[teste, alvo].to_numpy()
    cpl = custo_medio_celula(alvo)
    total = semanal[treino]
    previsoes = {
        "Baseline - custo médio geral": semanal.loc[teste, "investimento"] * total[alvo].sum() / total["investimento"].sum(),
        "Baseline - custo médio de cada praça x canal": semanal.loc[teste, "investimento"] / semanal.loc[teste, "praca_canal"].map(cpl),
        "Regressão de Poisson (curva por praça x canal)": poisson.predict(matriz_poisson(semanal[teste])),
        "Gradient Boosting monotônico": boosting.predict(matriz_boosting(semanal[teste])),
    }
    for nome, prev in previsoes.items():
        prev = np.asarray(prev, dtype=float)
        resultados.append({
            "alvo": alvo, "modelo": nome,
            "RMSE_semanal": np.sqrt(np.mean((real - prev) ** 2)),
            "erro_total_pct": np.abs(real - prev).sum() / real.sum() * 100,
        })

comparacao = pd.DataFrame(resultados)
print("\n============================")
print(f"PREVISÃO DAS ÚLTIMAS {SEMANAS_TESTE} SEMANAS (que os modelos não viram)")
print("============================")
print(comparacao.round(2).to_string(index=False))

## Situação atual de cada praça x canal

In [ ]:
atual = (
    semanal[teste].groupby("praca_canal")
    .agg(investimento_semana=("investimento", "mean"))
    .reset_index()
    .merge(semanal.drop_duplicates("praca_canal")[["praca_canal", "uf", "cidade", "canal",
                                                   "tipo_de_praca", "anuncios_mes"]],
           on="praca_canal")
)
x0 = atual["investimento_semana"].to_numpy()


def com_verba(x):
    df = atual.copy()
    df["log_investimento"] = np.log(x)
    return df


def resposta_poisson(alvo):
    return lambda x: modelos[alvo][0].predict(matriz_poisson(com_verba(x)))


def resposta_boosting(alvo):
    return lambda x: modelos[alvo][1].predict(matriz_boosting(com_verba(x)))


def custo_proximo(f, x, passo=0.10):
    """Custo do próximo resultado: quanto custa cada MQL a mais com +10% de verba."""
    return passo * x / np.maximum(f(x * (1 + passo)) - f(x), 1e-9)


if SIMULADO:
    curva = atual.merge(verdade[["cidade", "canal", "A", "K"]], on=["cidade", "canal"], how="left")
    DIAS_EFETIVOS = 5 * 1.1 + 2 * 0.75     # soma do efeito dia útil/fim de semana numa semana

    def resposta_verdadeira(alvo):
        taxa = (curva["tipo_de_praca"].map(TAXA_CADASTRO).to_numpy() if alvo == "cadastros"
                else np.ones(len(curva)))
        return lambda x: (curva["A"].to_numpy() * DIAS_EFETIVOS * (x / 7) / ((x / 7) + curva["K"].to_numpy()) * taxa)

## Validação: o modelo acerta o custo do próximo mql? (só na simulação)

In [ ]:
if SIMULADO:
    verdadeiro = custo_proximo(resposta_verdadeira(OBJETIVO), x0)
    acerto = pd.DataFrame([
        {"metodo": "Custo médio de cada praça x canal (o que se usa hoje)",
         "correlacao_com_o_verdadeiro": spearmanr(atual["praca_canal"].map(custo_medio_celula(OBJETIVO)), verdadeiro).correlation},
        {"metodo": "Regressão de Poisson",
         "correlacao_com_o_verdadeiro": spearmanr(custo_proximo(resposta_poisson(OBJETIVO), x0), verdadeiro).correlation},
        {"metodo": "Gradient Boosting monotônico",
         "correlacao_com_o_verdadeiro": spearmanr(custo_proximo(resposta_boosting(OBJETIVO), x0), verdadeiro).correlation},
    ])
    print("\n============================")
    print("CUSTO DO PRÓXIMO MQL: ESTIMADO x VERDADEIRO (simulação)")
    print("============================")
    print(acerto.round(3).to_string(index=False))
    print("Correlação: 1 = acerta perfeitamente a ordem das praças (da mais barata para a mais cara).")
    print("O custo médio também acerta a ordem, mas não diz QUANTO mover: ele não enxerga que cada")
    print("real a mais rende menos que o anterior. A Poisson enxerga, e é isso que dá o ganho abaixo.")

## Otimizador: distribuir o mesmo orçamento

In [ ]:
def otimizar(f, x_atual, orcamento, limite=LIMITE_VARIACAO, passos=PASSOS_OTIMIZADOR):
    """
    Começa com todas as praças x canais no mínimo permitido e vai entregando a verba,
    em pedaços, para quem gera mais resultado por real naquele momento, até acabar
    o orçamento. Cada praça x canal fica entre -limite e +limite da verba atual.
    """
    minimo, maximo = x_atual * (1 - limite), x_atual * (1 + limite)
    pedaco = (maximo - minimo) / passos
    grade = np.array([f(minimo + k * pedaco) for k in range(passos + 1)]).T
    posicao = np.zeros(len(x_atual), dtype=int)
    gasto = minimo.sum()
    indices = np.arange(len(x_atual))
    while True:
        proximo = np.minimum(posicao + 1, passos)
        ganho = np.where(posicao < passos, (grade[indices, proximo] - grade[indices, posicao]) / pedaco, -np.inf)
        j = int(np.argmax(ganho))
        if ganho[j] == -np.inf or gasto + pedaco[j] > orcamento + 1e-6:
            break
        posicao[j] += 1
        gasto += pedaco[j]
    x = minimo + posicao * pedaco
    return x * orcamento / x.sum()


def regra_simples(alvo, limite=LIMITE_VARIACAO):
    """Sem modelo: +limite para os 25% de menor custo médio, -limite para os 25% de maior."""
    cpl = atual["praca_canal"].map(custo_medio_celula(alvo)).to_numpy()
    baixo, alto = np.quantile(cpl, [0.25, 0.75])
    x = x0.copy()
    x[cpl <= baixo] *= 1 + limite
    x[cpl >= alto] *= 1 - limite
    return x * x0.sum() / x.sum()


orcamento = x0.sum()
f_poisson = resposta_poisson(OBJETIVO)
f_boosting = resposta_boosting(OBJETIVO)

planos = {
    "Regra simples (custo médio)": regra_simples(OBJETIVO),
    "Otimizador + Regressão de Poisson": otimizar(f_poisson, x0, orcamento),
    "Otimizador + Gradient Boosting": otimizar(f_boosting, x0, orcamento),
}
if SIMULADO:
    f_verdade = resposta_verdadeira(OBJETIVO)
    planos["Máximo possível (conhecendo a resposta certa)"] = otimizar(f_verdade, x0, orcamento)


def ganho(f, x):
    return f(x).sum() / f(x0).sum() - 1


print("\n============================")
print(f"MESMO ORÇAMENTO, NOVA DISTRIBUIÇÃO (limite de ±{LIMITE_VARIACAO:.0%} por praça x canal)")
print("============================")
resumo_planos = []
for nome, x in planos.items():
    linha = {"plano": nome, "ganho_previsto_pelo_poisson_pct": ganho(f_poisson, x) * 100}
    if SIMULADO:
        linha["ganho_verdadeiro_mqls_pct"] = ganho(resposta_verdadeira("mqls"), x) * 100
        linha["ganho_verdadeiro_cadastros_pct"] = ganho(resposta_verdadeira("cadastros"), x) * 100
    resumo_planos.append(linha)
resumo_planos = pd.DataFrame(resumo_planos)
print(resumo_planos.round(2).to_string(index=False))

MELHOR_PLANO = "Otimizador + Regressão de Poisson"
x_novo = planos[MELHOR_PLANO]

## Economia: menor orçamento que mantém os mqls de hoje

In [ ]:
meta = f_poisson(x0).sum()
baixo, alto = orcamento * (1 - LIMITE_VARIACAO), orcamento
for _ in range(40):
    meio = (baixo + alto) / 2
    if f_poisson(otimizar(f_poisson, x0, meio)).sum() >= meta:
        alto = meio
    else:
        baixo = meio
orcamento_minimo = alto
economia_semana = orcamento - orcamento_minimo

print(f"\nMesmos {OBJETIVO} de hoje gastando {orcamento_minimo / orcamento:.1%} do orçamento: "
      f"economia de R$ {economia_semana * SEMANAS_POR_MES:,.0f} por mês ({1 - orcamento_minimo / orcamento:.1%}).")
if SIMULADO:
    x_economia = otimizar(f_poisson, x0, orcamento_minimo)
    print(f"Conferência com a resposta certa: o plano econômico entrega "
          f"{resposta_verdadeira(OBJETIVO)(x_economia).sum() / resposta_verdadeira(OBJETIVO)(x0).sum():.1%} dos {OBJETIVO} de hoje.")

## Recomendação por praça x canal

In [ ]:
recomendacao = atual[["uf", "cidade", "canal", "tipo_de_praca"]].copy()
recomendacao["verba_mes_atual"] = x0 * SEMANAS_POR_MES
recomendacao["verba_mes_recomendada"] = x_novo * SEMANAS_POR_MES
recomendacao["variacao_pct"] = (x_novo / x0 - 1) * 100
recomendacao["custo_medio_por_mql"] = atual["praca_canal"].map(custo_medio_celula("mqls")).to_numpy()
recomendacao["custo_do_proximo_mql_hoje"] = custo_proximo(resposta_poisson("mqls"), x0)
recomendacao["custo_do_proximo_mql_depois"] = custo_proximo(resposta_poisson("mqls"), x_novo)
recomendacao["mqls_mes_atual"] = resposta_poisson("mqls")(x0) * SEMANAS_POR_MES
recomendacao["mqls_mes_depois"] = resposta_poisson("mqls")(x_novo) * SEMANAS_POR_MES
recomendacao["acao"] = np.select(
    [recomendacao["variacao_pct"] >= LIMIAR_ACAO * 100, recomendacao["variacao_pct"] <= -LIMIAR_ACAO * 100],
    ["AUMENTAR", "REDUZIR"], default="MANTER",
)
recomendacao = recomendacao.sort_values("variacao_pct", ascending=False)

print("\n============================")
print("RECOMENDAÇÃO POR PRAÇA x CANAL (10 maiores aumentos e 10 maiores cortes)")
print("============================")
colunas_ver = ["cidade", "canal", "tipo_de_praca", "verba_mes_atual", "verba_mes_recomendada",
               "custo_medio_por_mql", "custo_do_proximo_mql_hoje", "acao"]
print(recomendacao[colunas_ver].head(10).round(0).to_string(index=False))
print("...")
print(recomendacao[colunas_ver].tail(10).round(0).to_string(index=False))
print("\nAções:", recomendacao["acao"].value_counts().to_dict())

## Por tipo de praça: custo do próximo mql e para onde vai a verba

In [ ]:
def media_ponderada(df, coluna):
    return np.average(df[coluna], weights=df["verba_mes_atual"])


por_tipo = (
    recomendacao.groupby("tipo_de_praca")
    .apply(lambda g: pd.Series({
        "pracas_canais": len(g),
        "verba_mes_atual": g["verba_mes_atual"].sum(),
        "verba_mes_recomendada": g["verba_mes_recomendada"].sum(),
        "custo_medio_por_mql": g["verba_mes_atual"].sum() / g["mqls_mes_atual"].sum(),
        "custo_do_proximo_mql_hoje": media_ponderada(g, "custo_do_proximo_mql_hoje"),
    }), include_groups=False)
    .reset_index()
)
por_tipo["variacao_pct"] = (por_tipo["verba_mes_recomendada"] / por_tipo["verba_mes_atual"] - 1) * 100
por_tipo["fatia_da_verba_atual_pct"] = por_tipo["verba_mes_atual"] / por_tipo["verba_mes_atual"].sum() * 100
por_tipo = por_tipo.sort_values("custo_do_proximo_mql_hoje")

print("\n============================")
print("POR TIPO DE PRAÇA")
print("============================")
print(por_tipo.round(1).to_string(index=False))

# Investimento em praças de baixo retorno: próximo MQL custa mais que o dobro da mediana
limite_caro = 2 * np.median(recomendacao["custo_do_proximo_mql_hoje"])
caras = recomendacao[recomendacao["custo_do_proximo_mql_hoje"] > limite_caro]
print(f"\nVerba em praças x canais onde o próximo MQL custa mais que o dobro da mediana "
      f"(R$ {limite_caro:,.0f}): R$ {caras['verba_mes_atual'].sum():,.0f} por mês "
      f"({caras['verba_mes_atual'].sum() / recomendacao['verba_mes_atual'].sum():.1%} do total).")

## Sensibilidade ao limite de variação

In [ ]:
sensibilidade = []
for limite in [0.15, 0.30, 0.50]:
    linha = {"limite_por_praca_canal_pct": limite * 100}
    for nome, plano in [("regra_simples", regra_simples(OBJETIVO, limite)),
                        ("poisson", otimizar(f_poisson, x0, orcamento, limite))]:
        linha[f"ganho_previsto_{nome}_pct"] = ganho(f_poisson, plano) * 100
        if SIMULADO:
            linha[f"ganho_verdadeiro_{nome}_pct"] = ganho(f_verdade, plano) * 100
    if SIMULADO:
        linha["ganho_maximo_possivel_pct"] = ganho(f_verdade, otimizar(f_verdade, x0, orcamento, limite)) * 100
    sensibilidade.append(linha)
sensibilidade = pd.DataFrame(sensibilidade)

print("\nGanho de MQLs conforme o quanto cada praça x canal pode mudar:")
print(sensibilidade.round(2).to_string(index=False))

## Teste de estresse (só na simulação)

In [ ]:
# Pior caso: a saturação de cada praça não tem relação com nenhuma variável do modelo,
# e a verba quase não varia ao longo do ano. O modelo ainda acha o ganho?
if SIMULADO:
    estresse = []
    for rotulo, var_mes, var_dia, ligada in [
        ("cenário base", 0.25, 0.20, True),
        ("saturação aleatória", 0.25, 0.20, False),
        ("saturação aleatória + verba quase parada", 0.02, 0.01, False),
    ]:
        m_sim, v_sim = simular(SEMENTE + 1, var_mes, var_dia, ligada)
        m_sim["semana"] = m_sim["data"].dt.to_period("W").dt.start_time
        s = m_sim.groupby(["uf", "cidade", "canal", "semana"], as_index=False)[["investimento", "mqls"]].sum()
        s = s.merge(v_sim[["cidade", "canal", "tipo_de_praca", "A", "K"]], on=["cidade", "canal"])
        s["chave"] = normalizar_nome(s["cidade"])
        s = s.merge(perfil_cidades[["uf", "chave", "anuncios_mes"]], on=["uf", "chave"], how="left")
        s["praca_canal"] = s["cidade"] + " | " + s["canal"]
        s["log_investimento"] = np.log(s["investimento"])
        sem = sorted(s["semana"].unique())
        tr_s = s["semana"] < sem[-SEMANAS_TESTE]
        modelo_s = PoissonRegressor(alpha=1e-5, max_iter=5000).fit(matriz_poisson(s[tr_s]), s.loc[tr_s, "mqls"])
        at = s[~tr_s].groupby("praca_canal").agg(investimento=("investimento", "mean")).reset_index()
        at = at.merge(s.drop_duplicates("praca_canal").drop(columns=["investimento", "mqls", "semana"]), on="praca_canal")
        xa = at["investimento"].to_numpy()

        def f_m(x, at=at, modelo_s=modelo_s):
            d = at.copy()
            d["log_investimento"] = np.log(x)
            return modelo_s.predict(matriz_poisson(d))

        def f_v(x, at=at):
            return at["A"].to_numpy() * DIAS_EFETIVOS * (x / 7) / ((x / 7) + at["K"].to_numpy())

        cpl_s = (s[tr_s].groupby("praca_canal")["investimento"].sum()
                 / s[tr_s].groupby("praca_canal")["mqls"].sum().clip(lower=1))
        q = at["praca_canal"].map(cpl_s).to_numpy()
        a, b = np.quantile(q, [0.25, 0.75])
        xr = xa.copy()
        xr[q <= a] *= 1 + LIMITE_VARIACAO
        xr[q >= b] *= 1 - LIMITE_VARIACAO
        xr *= xa.sum() / xr.sum()
        base_v = f_v(xa).sum()
        estresse.append({
            "cenario": rotulo,
            "ganho_regra_simples_pct": (f_v(xr).sum() / base_v - 1) * 100,
            "ganho_poisson_pct": (f_v(otimizar(f_m, xa, xa.sum())).sum() / base_v - 1) * 100,
            "ganho_maximo_pct": (f_v(otimizar(f_v, xa, xa.sum())).sum() / base_v - 1) * 100,
        })
    estresse = pd.DataFrame(estresse)
    print("\n============================")
    print("TESTE DE ESTRESSE (ganho verdadeiro de MQLs)")
    print("============================")
    print(estresse.round(2).to_string(index=False))

## Salvar

In [ ]:
comparacao.to_csv(TABLES_DIR / f"realocacao_comparacao_modelos{SUFIXO}.csv", index=False)
resumo_planos.to_csv(TABLES_DIR / f"realocacao_planos{SUFIXO}.csv", index=False)
recomendacao.to_csv(TABLES_DIR / f"realocacao_recomendacao_praca_canal{SUFIXO}.csv", index=False)
por_tipo.to_csv(TABLES_DIR / f"realocacao_por_tipo_de_praca{SUFIXO}.csv", index=False)
sensibilidade.to_csv(TABLES_DIR / f"realocacao_sensibilidade_limites{SUFIXO}.csv", index=False)
if SIMULADO:
    acerto.to_csv(TABLES_DIR / "realocacao_acerto_custo_proximo_SIMULADO.csv", index=False)
    estresse.to_csv(TABLES_DIR / "realocacao_teste_de_estresse_SIMULADO.csv", index=False)

print("\nTabelas salvas em:", TABLES_DIR)

## Gráfico: ganho de cada plano

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
coluna = "ganho_verdadeiro_mqls_pct" if SIMULADO else "ganho_previsto_pelo_poisson_pct"
ordem = resumo_planos.sort_values(coluna)
cores = ["#3d5a80" if "Máximo" in p else "#1f6f5c" if "Poisson" in p else "#e07a5f" if "Boosting" in p else "#b8b8b8"
         for p in ordem["plano"]]
ax.barh(ordem["plano"], ordem[coluna], color=cores)
for y_pos, valor in enumerate(ordem[coluna]):
    ax.text(valor, y_pos, f" {valor:+.1f}%", va="center", fontsize=10)
ax.set_xlabel("MQLs a mais com o mesmo orçamento (%)" + (" - resultado verdadeiro da simulação" if SIMULADO else " - previsto"))
ax.set_title(f"Quanto cada plano de realocação ganha (limite de ±{LIMITE_VARIACAO:.0%} por praça x canal)")
ax.axvline(0, color="#555", linewidth=0.8)
ax.set_xlim(min(0, ordem[coluna].min()) * 1.35, ordem[coluna].max() * 1.25)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"realocacao_ganho_planos{SUFIXO}.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## Gráfico: custo médio x custo do próximo mql

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
for tipo, cor in zip(tipos, ["#1f6f5c", "#e07a5f", "#3d5a80", "#c9a227", "#888888", "#81b29a"]):
    parte = recomendacao[recomendacao["tipo_de_praca"] == tipo]
    ax.scatter(parte["custo_medio_por_mql"], parte["custo_do_proximo_mql_hoje"], s=np.sqrt(parte["verba_mes_atual"]) / 3,
               color=cor, alpha=0.7, edgecolor="white", label=tipo)
limite_eixo = [recomendacao["custo_medio_por_mql"].min() * 0.9, recomendacao["custo_medio_por_mql"].max() * 1.1]
ax.plot(limite_eixo, limite_eixo, linestyle="--", color="#555", linewidth=1, label="próximo = médio")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Custo médio por MQL (R$) - o que se olha hoje")
ax.set_ylabel("Custo do PRÓXIMO MQL (R$) - estimado pelo modelo")
ax.set_title("O próximo MQL custa mais que a média em todas as praças")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"realocacao_custo_medio_vs_proximo{SUFIXO}.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## Gráfico: verba atual x recomendada por tipo de praça

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
posicoes = np.arange(len(por_tipo))
ax.barh(posicoes - 0.2, por_tipo["verba_mes_atual"] / 1000, height=0.4, color="#b8b8b8", label="Verba atual")
ax.barh(posicoes + 0.2, por_tipo["verba_mes_recomendada"] / 1000, height=0.4, color="#1f6f5c", label="Verba recomendada")
for p, (_, linha) in zip(posicoes, por_tipo.iterrows()):
    ax.text(max(linha["verba_mes_atual"], linha["verba_mes_recomendada"]) / 1000, p, f"  {linha['variacao_pct']:+.0f}%",
            va="center", fontsize=9)
ax.set_yticks(posicoes)
ax.set_yticklabels(por_tipo["tipo_de_praca"])
ax.set_xlabel("Verba por mês (R$ mil)")
ax.set_title("Para onde vai a verba, por tipo de praça")
ax.set_xlim(0, por_tipo[["verba_mes_atual", "verba_mes_recomendada"]].to_numpy().max() / 1000 * 1.15)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"realocacao_verba_por_tipo{SUFIXO}.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## Conclusão

In [ ]:
p = resumo_planos.set_index("plano")
print("\n============================")
print("CONCLUSÃO - REALOCAÇÃO DA VERBA" + (" (DADOS SIMULADOS)" if SIMULADO else ""))
print("============================")
if SIMULADO:
    print(
        f"1) Com o mesmo orçamento, o otimizador com a Regressão de Poisson gera "
        f"{p.loc[MELHOR_PLANO, 'ganho_verdadeiro_mqls_pct']:+.1f}% de MQLs de verdade, contra "
        f"{p.loc['Regra simples (custo médio)', 'ganho_verdadeiro_mqls_pct']:+.1f}% da regra simples e "
        f"{p.loc['Máximo possível (conhecendo a resposta certa)', 'ganho_verdadeiro_mqls_pct']:+.1f}% do máximo possível."
    )
    print(
        f"2) O Gradient Boosting ficou em {p.loc['Otimizador + Gradient Boosting', 'ganho_verdadeiro_mqls_pct']:+.1f}%: "
        "árvores fazem 'degraus', e o custo do próximo MQL precisa de uma curva suave. "
        "Ele também previu pior as semanas de teste. Para decidir verba, a curva da Poisson é melhor."
    )
    print(
        "3) O ganho resiste ao teste de estresse: mesmo com a saturação sem relação com as variáveis "
        "e a verba quase parada, o modelo fica perto do máximo, porque estima as duas coisas que "
        "importam com limite de ±30%: a eficiência de cada praça x canal e o retorno decrescente."
    )
else:
    print(f"1) Com o mesmo orçamento, o plano recomendado prevê {p.loc[MELHOR_PLANO, 'ganho_previsto_pelo_poisson_pct']:+.1f}% de MQLs.")
print(
    f"4) Economia: os mesmos {OBJETIVO} de hoje com {1 - orcamento_minimo / orcamento:.1%} menos verba "
    f"(R$ {economia_semana * SEMANAS_POR_MES:,.0f} por mês)."
)
print(
    f"5) Verba em praças x canais de baixo retorno (próximo MQL acima do dobro da mediana): "
    f"{caras['verba_mes_atual'].sum() / recomendacao['verba_mes_atual'].sum():.1%} do total."
)
print(
    "\nPara usar com os dados reais: preencha um arquivo no formato da planilha-modelo "
    "(realocacao_modelo_planilha_loft.csv), coloque na pasta DATA_DIR e defina MIDIA_FILE no .env. "
    "Para medir o efeito causal antes de mover a verba toda, combine com o teste geográfico."
)